In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sparrow as sp
import spatialdata as sd
import pandas as pd
import numpy as np
import dask as da
import os
import gc
import geopandas as gpd
from matplotlib import pyplot as plt
import seaborn as sns
import h5py
import json

from napari_spatialdata import Interactive

from spatialdata_io.readers.visium import visium

from spatialdata.transformations import get_transformation
from sparrow.image._image import _get_spatial_element

import scanpy as sc
from sparrow.table import add_table_layer
from sparrow.points import add_points_layer
from sparrow.shape import add_shapes_layer
from sparrow.utils._keys import _INSTANCE_KEY, _REGION_KEY

import dask.dataframe as dd

from skimage import io

the value of the environment variable BASIC_DCT_BACKEND is not in ["JAX","SCIPY"]
2024-11-08 16:40:47,054 - sparrow.image.pixel_clustering._clustering - WARNING - 'flowsom' not installed, to use 'sp.im.flowsom', please install this library.
2024-11-08 16:40:48,310 - sparrow.table.cell_clustering._clustering - WARNING - 'flowsom' not installed, to use 'sp.tb.flowsom', please install this library.
2024-11-08 16:40:48,378 - sparrow.plot._qc_cells - WARNING - 'joypy' not installed, to use 'sp.pl.ridgeplot_channel' and 'sp.pl.ridgeplot_channel_sample', please install this library.
2024-11-08 16:40:48,380 - sparrow.plot._qc_image - WARNING - 'textalloc' not installed, to use 'sp.pl.group_snr_ratio' and 'sp.pl.snr_ratio', please install this library.


## Create sdata

In [3]:
sdata = visium(
    path=r'D:\Data\SCA013\outs', 
    counts_file = "raw_feature_bc_matrix.h5",
    dataset_id = "_")
sdata

INFO     reading D:\Data\SCA013\outs\raw_feature_bc_matrix.h5                                                      


c:\Users\julienm\miniconda3\envs\harpy\lib\site-packages\anndata\_core\anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
c:\Users\julienm\miniconda3\envs\harpy\lib\site-packages\anndata\_core\anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
c:\Users\julienm\miniconda3\envs\harpy\lib\site-packages\spatialdata\models\models.py:1018: UserWarning: Converting `region_key: region` to categorical dtype.
  return convert_region_column_to_categorical(adata)


SpatialData object
├── Images
│     ├── '__hires_image': DataArray[cyx] (3, 1647, 2000)
│     └── '__lowres_image': DataArray[cyx] (3, 494, 600)
├── Shapes
│     └── '_': GeoDataFrame shape: (4946, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (4946, 32285)
with coordinate systems:
    ▸ 'downscaled_hires', with elements:
        __hires_image (Images), _ (Shapes)
    ▸ 'downscaled_lowres', with elements:
        __lowres_image (Images), _ (Shapes)
    ▸ 'global', with elements:
        _ (Shapes)

In [4]:
probe_set_df = pd.read_csv(r"d:\Data\SCA013\outs\probe_set.csv", comment='#')
included = set(probe_set_df[probe_set_df['included'] == True]['gene_id'])
adata = sdata.tables['table'][:, sdata.tables['table'].var['gene_ids'].isin(included)]

In [5]:
print(f"Number of spots in tissue: {(adata.obs['in_tissue'] == 1).sum()}")

Number of spots in tissue: 1937


In [6]:
print(f"Total number of genes: {(adata.X.sum(axis=0) > 0).sum()}")

Total number of genes: 19027


In [7]:
print(f"Number of genes in tissue: {(adata[adata.obs['in_tissue'] == 1].X.sum(axis=0) > 0).sum()}")

Number of genes in tissue: 19027


In [8]:
print(f"Total number of UMIs: {adata.X.sum()}")

Total number of UMIs: 21475076.0


In [9]:
print(f"Number of UMIs in tissue: {adata[adata.obs['in_tissue'] == 1].X.sum()}")

Number of UMIs in tissue: 21068796.0


In [10]:
import h5py
import numpy as np

# Open the HDF5 file
with h5py.File(r"D:\Data\SCA013\outs\molecule_info.h5", "r") as f:
    # Load the feature_id, feature_idx, and count datasets
    feature_id = f['/features/id'][:]
    feature_idx = f['/feature_idx'][:]
    count = f['/count'][:]

    # Decode feature_id entries
    feature_id = np.array([i.decode('ascii') for i in feature_id])
    
    # Find indices to keep
    included_feature_mask = np.isin(feature_id, list(included))
    included_feature_indices = np.where(included_feature_mask)[0]
    included_counts_mask = np.isin(feature_idx, included_feature_indices)
    
    # Sum counts for filtered indices
    included_counts = count[included_counts_mask]
    total_sum = np.sum(included_counts)
    total_count = len(included_counts)

    print("Total number of UMIs for included genes:", total_sum)
    print("Number of unique UMIs for included genes:", total_count)


Total number of UMIs for included genes: 38684688
Number of unique UMIs for included genes: 21475076


In [11]:
1-(total_count/total_sum)

0.44486883285707257